# Locate SNOTEL Station & Download Forcing (snow example)

This notebook helps you:
1. Pick a SNOTEL snow-pillow site from the showcase catalog (or enter your own)
2. Query CONUS2 subsurface parameters at the site location
3. Prepare CLM input files (vegm, clmin)
4. Download CW3E meteorological forcing
5. Preview the SNOTEL SWE observations the comparison notebook will use

**Prerequisites:** HydroData account (free) — see README.md for signup instructions.


In [ ]:
# Imports
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import subsettools as st
import hf_hydrodata as hf
import shutil
from pathlib import Path
from datetime import datetime

# Make helpers.py (at repo root) importable from this subdir
sys.path.insert(0, os.path.abspath(".."))
from helpers import (IGBP_NAMES, IGBP_FULL_NAMES,
                     CONUS2_SOILS, CONUS2_GEOLOGY, CONUS2_SUBSURFACE,
                     SNOTEL_SITES, SNOTEL_SHOWCASE,
                     build_vegm, patch_clmin_dates, write_site_config)


In [ ]:
# ============================================================
# User Configuration
# ============================================================
# Pick a SNOTEL site + water year. Defaults to a showcase entry.

site_name  = "Berthoud_Summit"     # key into SNOTEL_SITES (see below)
water_year = 2024                  # Oct 1 (WY-1) to Oct 1 (WY)

start = f"{water_year - 1}-10-01"
end   = f"{water_year}-10-01"

# Directory where forcing and CLM input files will be written
# (path is relative to this notebook's directory, i.e. snow/)
static_write_dir = f"../runs/snow/{site_name}_WY{water_year}"

# Forcing filename
forcing_filename = f"forcing1D.{site_name}.{start}-{end}.txt"

# HydroData PIN (only needed once per machine)
# Sign up: https://hydrogen.princeton.edu/signup
# Get PIN: https://hydrogen.princeton.edu/pin
hf.register_api_pin("your_email@example.com", "your_pin")

# -----------------------------------------------------------
# Showcase sites (geographically diverse, recent CW3E-covered years)
# -----------------------------------------------------------
print("Showcase site-years:")
for s, wy in SNOTEL_SHOWCASE:
    info = SNOTEL_SITES[s]
    print(f"  {s:<18s} WY{wy}  {info['region']:<18s}  {info['elev_m']} m  triplet={info['triplet']}")

# -----------------------------------------------------------
# Full SNOTEL catalog: see helpers.SNOTEL_SITES for the 12 curated sites.
# To extend, add entries following the same schema — coords, elev_m, state,
# region, triplet. The low-bias site-years from snow_model_sensitivity
# (|T bias|<1C, |P bias|<10%) are documented in the repo README.
# -----------------------------------------------------------


In [ ]:
# Resolve site metadata
if site_name not in SNOTEL_SITES:
    raise KeyError(f"{site_name} not in SNOTEL_SITES — add it to helpers.SNOTEL_SITES")

site = SNOTEL_SITES[site_name]
lat, lon = site["coords"]
elev_m   = site["elev_m"]
region   = site["region"]
triplet  = site["triplet"]

print(f"Site: {site_name} ({region}, {site['state']})")
print(f"  lat={lat:.4f}, lon={lon:.4f}, elev={elev_m} m")
print(f"  SNOTEL triplet: {triplet}")


## CONUS2 Grid & Subsurface Parameters

Snow ET sensitivity is not especially sensitive to subsurface — but we still
need sensible soil/geology for the ParFlow domain. Query CONUS2 to populate
them; water table depth defaults to deep (well below the 8m domain) since
SNOTEL sites are mountain ridges where deep water tables are the norm.


In [ ]:
# Define single-column CONUS2 domain (single point = single CONUS2 cell)
latlon_bounds = [[lat, lon], [lat, lon]]
ij_bounds, mask = st.define_latlon_domain(latlon_bounds=latlon_bounds, grid="conus2")
print(f"CONUS2 grid bounds (i_start, j_start, i_end, j_end): {ij_bounds}")

# Query full pf_indicator profile (10 layers, bottom to top)
ind = hf.get_gridded_data({
    "dataset": "conus2_domain", "variable": "pf_indicator",
    "grid_bounds": ij_bounds,
})
print(f"\nCONUS2 subsurface profile (bottom -> top):")
for i in range(ind.shape[0]):
    t = int(ind[i, 0, 0])
    info = CONUS2_SUBSURFACE.get(t, ("Unknown",))
    print(f"  Layer {i}: type {t} ({info[0]})")

# Soil = top layer, Geology = layer 4 (from bottom)
soil_type = int(ind[-1, 0, 0])
geol_type = int(ind[4, 0, 0])

soil_info = CONUS2_SOILS.get(soil_type)
geol_info = CONUS2_GEOLOGY.get(geol_type)

print(f"\n--- Soil (top layer, type {soil_type}: {soil_info[0]}) ---")
print(f"  Ksat={soil_info[1]} m/h, porosity={soil_info[2]}, "
      f"VG alpha={soil_info[3]}, VG n={soil_info[4]}, Sres={soil_info[5]}")
if geol_info:
    print(f"\n--- Geology (layer 4, type {geol_type}: {geol_info[0]}) ---")
    print(f"  Ksat={geol_info[1]} m/h, porosity={geol_info[2]}, "
          f"VG alpha={geol_info[3]}, VG n={geol_info[4]}, Sres={geol_info[5]}")


In [ ]:
## Water table depth — default deep for mountain snow sites.
## Override here if you have site-specific WTD (negative = below surface).
wtd_m = -10.0  # m — below the 8m domain; snow is rainfall-dominated at depth
print(f"WTD BC: {wtd_m} m")


In [ ]:
# ============================================================
# Vegm parameters — forced to IGBP 7 (shrubland) for snow default config
# ============================================================
# The snow sensitivity study found shrubland IGBP produces the best SWE
# timing across sites. Override if a site has a known different land cover.
igbp_type = 7  # Open Shrubland — best snow config default

# Sand/clay: try CONUS2 config_clm, fall back to defaults
import tempfile
_tmp_dir = tempfile.mkdtemp(prefix="clm_tmp_")
try:
    _ = st.config_clm(ij_bounds, start=start, end=end,
                      dataset="conus2_domain", write_dir=_tmp_dir)
    with open(os.path.join(_tmp_dir, "drv_vegm.dat")) as _f:
        _lines = _f.readlines()
    for _line in _lines:
        _parts = _line.split()
        if len(_parts) >= 25:
            sand_frac = float(_parts[4])
            clay_frac = float(_parts[5])
            color_index = int(_parts[6])
            break
    print("Sand/clay from CONUS2 domain")
except Exception as e:
    print(f"config_clm failed ({e}), using defaults")
    sand_frac, clay_frac, color_index = 0.40, 0.20, 4

print(f"\n--- Vegetation / Soil Texture ---")
print(f"  IGBP:        {igbp_type} — {IGBP_FULL_NAMES[igbp_type]} ({IGBP_NAMES[igbp_type]})")
print(f"  Sand:        {sand_frac:.0%}")
print(f"  Clay:        {clay_frac:.0%}")
print(f"  Soil color:  {color_index}")


## Prepare Run Directory

Write CLM input files (drv_vegm.dat, drv_clmin.dat, drv_vegp.dat) to the run directory.


In [ ]:
# Create run directory and write CLM input files
run_dir = Path(static_write_dir)
run_dir.mkdir(parents=True, exist_ok=True)

# 1. drv_vegm.dat (shrubland = IGBP 7)
build_vegm(run_dir / "drv_vegm.dat", lat, lon, sand_frac, clay_frac, color_index, igbp_type)
print(f"Wrote drv_vegm.dat (IGBP={igbp_type}, shrubland)")

# 2. drv_clmin.dat (shared template, patched dates)
clm_inputs_shared = Path("../clm_inputs")
patch_clmin_dates(clm_inputs_shared / "drv_clmin_template.dat",
                  run_dir / "drv_clmin.dat", water_year)
print(f"Wrote drv_clmin.dat (WY{water_year})")

# 3. drv_vegp.dat (snow-specific: copied from snow_model_sensitivity reference)
clm_inputs_snow = Path("../clm_inputs/snow")
shutil.copy2(clm_inputs_snow / "drv_vegp.dat", run_dir / "drv_vegp.dat")
print(f"Copied drv_vegp.dat (clm_snow_improved config)")

print(f"\nRun directory: {run_dir}")
print("Contents:", [f.name for f in sorted(run_dir.iterdir())])


## Download CW3E Forcing

Download 8 meteorological forcing variables from CW3E via HydroData.


In [ ]:
# CW3E forcing in ParFlow-CLM 1D order:
#   DSWR, DLWR, APCP [mm/s], Temp [K], UGRD, VGRD, Press [Pa], SPFH [kg/kg]
forcing_vars = [
    "downward_shortwave", "downward_longwave", "precipitation",
    "air_temp", "east_windspeed", "north_windspeed",
    "atmospheric_pressure", "specific_humidity",
]

dt_start = datetime.strptime(start, "%Y-%m-%d")
dt_end   = datetime.strptime(end,   "%Y-%m-%d")
n_hours  = int((dt_end - dt_start).total_seconds() / 3600)
print(f"Downloading {n_hours} hours for {site_name}...")

forcing_data = np.zeros((n_hours, 8))
for i, var in enumerate(forcing_vars):
    print(f"  [{i+1}/8] {var}...", end=" ", flush=True)
    options = {
        "dataset": "CW3E", "period": "hourly", "variable": var,
        "start_time": start, "end_time": end,
        "grid_bounds": ij_bounds, "dataset_version": "1.0",
    }
    data = hf.get_gridded_data(options)
    col = data[:, 0, 0] if data.ndim == 3 else data.flatten()
    if len(col) != n_hours:
        col = col[:n_hours] if len(col) > n_hours else np.pad(col, (0, n_hours - len(col)))
    forcing_data[:, i] = col
    print(f"OK (range: {col.min():.3g} to {col.max():.3g})")

forcing_path = run_dir / forcing_filename
np.savetxt(forcing_path, forcing_data, fmt="%.6e", delimiter=" ")
print(f"\nSaved: {forcing_path} ({forcing_data.shape[0]} hours, "
      f"{forcing_path.stat().st_size/1024:.0f} KB)")


In [ ]:
# Plot forcing: precipitation and temperature
forcing_data = np.loadtxt(run_dir / forcing_filename)
n_hours = forcing_data.shape[0]
dates = pd.date_range(start, periods=n_hours, freq="h")

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(dates, forcing_data[:, 2] * 3600, color="steelblue", lw=0.5)
axes[0].set_ylabel("Precip [mm/h]")
axes[0].set_title(f"{site_name} CW3E Forcing (WY{water_year})")

axes[1].plot(dates, forcing_data[:, 3] - 273.15, color="firebrick", lw=0.3)
axes[1].axhline(0, color="gray", lw=0.5, ls="--")
axes[1].set_ylabel("Air Temp [C]")
axes[1].set_xlabel("Date")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.show()


## Preview SNOTEL SWE Observations

Fetch the daily SWE time series the comparison notebook will use.


In [ ]:
# Fetch SNOTEL SWE for preview
try:
    df = hf.get_point_data(
        dataset="snotel", variable="swe",
        temporal_resolution="daily", aggregation="sod",
        site_ids=[triplet],
        date_start=start, date_end=end,
    )
    if df is None or df.empty or triplet not in df.columns:
        print(f"No SNOTEL SWE data returned for triplet {triplet}")
        obs_swe = None
    else:
        obs_dates = pd.to_datetime(df["date"])
        obs_swe = pd.Series(df[triplet].values.astype(float),
                            index=pd.DatetimeIndex(obs_dates), name="swe_mm").dropna()
        print(f"SNOTEL SWE: {len(obs_swe)} daily values, "
              f"peak={obs_swe.max():.0f} mm, mean={obs_swe.mean():.0f} mm")
except Exception as e:
    print(f"SNOTEL fetch failed: {e}")
    obs_swe = None

if obs_swe is not None and len(obs_swe) > 0:
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(obs_swe.index, obs_swe.values, color="steelblue", lw=1)
    ax.set_ylabel("SWE [mm]")
    ax.set_title(f"{site_name} WY{water_year} — SNOTEL SWE observations")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


## Write Site Configuration

Save all site parameters to a config file that Notebooks 2 and 3 will load.


In [ ]:
# Write site_config.txt
config_path = run_dir / "site_config.txt"
s = CONUS2_SOILS[soil_type]
params = {
    "site_name":  site_name,
    "water_year": water_year,
    "start":      start,
    "end":        end,
    "lat":        f"{lat:.4f}",
    "lon":        f"{lon:.4f}",
    "elev_m":     elev_m,
    "region":     region,
    "triplet":    triplet,
    "igbp":       igbp_type,
    "utc_offset": round(lon / 15),
    "_comment_soil": f"Soil (CONUS2 top layer, type {soil_type}: {s[0]})",
    "soil_ksat": s[1],
    "soil_porosity": s[2],
    "soil_vg_alpha": s[3],
    "soil_vg_n": s[4],
    "soil_sres": s[5],
}
if geol_info:
    g = geol_info
    params["_comment_geol"] = f"Geology (CONUS2 layer 4, type {geol_type}: {g[0]})"
    params["geol_ksat"]     = g[1]
    params["geol_porosity"] = g[2]
    params["geol_vg_alpha"] = g[3]
    params["geol_vg_n"]     = g[4]
    params["geol_sres"]     = g[5]

params["_comment_wtd"] = "Water table depth (negative = below surface)"
params["wtd_m"] = wtd_m
params["_comment_forcing"] = "Forcing"
params["forcing_file"] = forcing_filename

write_site_config(config_path, params)
print(f"Wrote: {config_path}\n")
print(open(config_path).read())


Proceed to **Single_Column_PFCLM_netcdf.ipynb** — it will load `site_config.txt` and display the parameters for you to review before running.
